# 05 — Teoria dei fenotipi su Track#3
Replica di **EEG_16/16b** (subject clustering): ogni soggetto ha una **firma di connettività**
(matrice di adiacenza media tra elettrodi). Clusterizzando le firme, i soggetti si dividono in
**fenotipi** (nella tesi: C0 *fronto-motor* vs C1 *fronto-occipital*).

**Domande**: (1) i 15 soggetti Track#3 si separano in fenotipi di connettività? (2) il fenotipo **predice la decodabilità**? (3) la separazione è reale o guidata da qualità segnale?

> ⚠️ **Caveat**: 15 soggetti sono POCHI (la tesi ne aveva 74). Montaggio diverso (64ch vs 61ch).
> Questo è un test **esplorativo**, non una replica definitiva.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, matplotlib.pyplot as plt
import track3_config as C, track3_io as io, track3_train as T
import track3_phenotypes as PH
print(C.summary())
assert C.DATA_ROOT is not None, C._no_data_msg()

## §1 — Firme di connettività per soggetto (PLV e PCC)
Firma = media sui trial della connettività per-trial, triangolo superiore (2016 feature per 64 canali).
`prune_k` opzionale replica la pipeline pruned della tesi.

In [ ]:
feats_plv, ids, mats_plv = PH.subject_fingerprints('plv')      # broadband
feats_pcc, _,  mats_pcc = PH.subject_fingerprints('pcc')
print('PLV firme:', feats_plv.shape, '| PCC firme:', feats_pcc.shape, '| soggetti:', ids)

## §2 — Quanti fenotipi? Silhouette + clustering k=2 (PLV)

In [ ]:
sil = PH.silhouette_scan(feats_plv, ks=(2,3,4,5))
print('silhouette per k:', {k: round(v,3) for k,v in sil.items()})
labels, Z, pca, _ = PH.cluster_subjects(feats_plv, k=2)
print('cluster sizes:', np.bincount(labels), '| PCA var PC1-2:', pca.explained_variance_ratio_[:2].round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4.5))
ax[0].bar(list(sil.keys()), list(sil.values()), color='steelblue')
ax[0].set_xlabel('k'); ax[0].set_ylabel('silhouette'); ax[0].set_title('Silhouette vs k (PLV)')
for c in (0,1):
    m = labels==c
    ax[1].scatter(Z[m,0], Z[m,1], s=80, label=f'C{c} (n={m.sum()})', edgecolors='white')
for i,s in enumerate(ids):
    ax[1].annotate(f'S{s:02d}', (Z[i,0], Z[i,1]), fontsize=7, xytext=(3,3), textcoords='offset points')
ax[1].set_xlabel('PC1'); ax[1].set_ylabel('PC2'); ax[1].set_title('Fenotipi PLV (PCA 2D)'); ax[1].legend()
plt.tight_layout(); plt.savefig(C.FIG_DIR/'pheno_clustering_plv.png', dpi=130); plt.show()

## §3 — Concordanza PLV vs PCC (ARI)
Se i due metrici danno gli stessi cluster → la separazione è robusta, non un artefatto della metrica.

In [ ]:
from sklearn.metrics import adjusted_rand_score
labels_pcc, _, _, _ = PH.cluster_subjects(feats_pcc, k=2)
ari = adjusted_rand_score(labels, labels_pcc)
print(f'ARI PLV vs PCC = {ari:.3f}  (1.0=identici, 0=casuali)')
print('PLV :', labels)
print('PCC :', labels_pcc)

## §4 — Dove differiscono i fenotipi? (topomap C1 − C0)
Per ogni elettrodo: forza di connettività C1 − C0. Mostra quale regione separa i due fenotipi.

In [ ]:
import mne
clab, pos = io.canonical_positions()
info = mne.create_info(clab, C.FS, ch_types='eeg')
montage = mne.channels.make_dig_montage(ch_pos={c:p for c,p in zip(clab,pos)}, coord_frame='head')
info.set_montage(montage, on_missing='warn')
ns_diff = PH.node_strength_diff(mats_plv, labels)   # (64,)
fig, ax = plt.subplots(1, 3, figsize=(13,4))
for a,(title, vals) in zip(ax, [('C0 mean', mats_plv[labels==0].mean(0).mean(1)),
                                ('C1 mean', mats_plv[labels==1].mean(0).mean(1)),
                                ('C1 - C0', ns_diff)]):
    mne.viz.plot_topomap(vals, info, axes=a, show=False, cmap='RdBu_r')
    a.set_title(title)
fig.suptitle('Firma spaziale dei fenotipi (PLV node strength)')
plt.tight_layout(); plt.savefig(C.FIG_DIR/'pheno_topomap_plv.png', dpi=130); plt.show()
top_c1 = np.argsort(ns_diff)[::-1][:6];  top_c0 = np.argsort(ns_diff)[:6]
print('C1 più connesso:', [clab[i] for i in top_c1])
print('C0 più connesso:', [clab[i] for i in top_c0])

## §5 — Coppie discriminanti (Cohen's d) + permutation test
Quali connessioni electrode-pair separano i fenotipi, e la separazione è genuina (non circular analysis)?

In [ ]:
d = PH.cohens_d_pairs(feats_plv, labels)
triu = np.triu_indices(C.N_CHANNELS, k=1)
order = np.argsort(np.abs(d))[::-1][:20]
pairs = [f'{clab[triu[0][i]]}-{clab[triu[1][i]]}' for i in order]
fig, ax = plt.subplots(figsize=(9,5))
colors = ['crimson' if d[i]>0 else 'steelblue' for i in order]
ax.barh(range(len(order)), d[order], color=colors)
ax.set_yticks(range(len(order))); ax.set_yticklabels(pairs, fontsize=8); ax.invert_yaxis()
ax.set_xlabel("Cohen's d (>0: C0 più forte)"); ax.set_title('Top 20 coppie discriminanti (PLV)')
ax.axvline(0, color='k', lw=0.8)
plt.tight_layout(); plt.savefig(C.FIG_DIR/'pheno_cohensd_plv.png', dpi=130); plt.show()
obs, p = PH.permutation_test(feats_plv, labels, n_perm=5000)
print(f'Permutation test: max|d|={obs:.2f}  p_perm={p:.4f}  '
      + ('GENUINO (non circular)' if p<0.05 else 'non significativo'))

## §6 — Quality-check: la separazione è artefatto di qualità segnale?
Se C0 e C1 differiscono in varianza o potenza gamma → la separazione potrebbe essere driven da
artefatti/SNR, non da connettività funzionale. Vogliamo che NON differiscano troppo.

In [ ]:
from scipy.signal import welch
from scipy.stats import mannwhitneyu
var_s, gamma_s = [], []
for s in ids:
    tr,va,te = io.load_subject_all(s)
    X = np.concatenate([tr.X, va.X, te.X], 0)
    var_s.append(X.var(axis=(1,2)).mean())
    f, pxx = welch(X, fs=tr.fs, nperseg=256, axis=2)
    gmask = (f>=30)&(f<=45)
    gamma_s.append(pxx[:,:,gmask].mean())
var_s, gamma_s = np.array(var_s), np.array(gamma_s)
for name, arr in [('varianza', var_s), ('gamma 30-45', gamma_s)]:
    u,pv = mannwhitneyu(arr[labels==0], arr[labels==1])
    print(f'{name}: C0={arr[labels==0].mean():.3g} vs C1={arr[labels==1].mean():.3g}  p(MW)={pv:.3f}'
          + ('  ⚠️ differiscono (possibile SNR)' if pv<0.05 else '  ✓ non differiscono'))

## §7 — Il fenotipo predice la DECODABILITÀ?
Domanda chiave della tesi: i soggetti di un fenotipo si decodificano meglio?
Decodabilità per soggetto = LDA subject-dependent (veloce). Correlazione con il cluster.

In [ ]:
dec = T.classical_baseline(protocol='dependent', verbose=False)['per_subject']
dec = np.array(dec)
from scipy.stats import mannwhitneyu, pointbiserialr
u, pv = mannwhitneyu(dec[labels==0], dec[labels==1])
r, pr = pointbiserialr(labels, dec)
print(f'Decodabilità (LDA): C0={dec[labels==0].mean():.3f}  C1={dec[labels==1].mean():.3f}  p(MW)={pv:.3f}')
print(f'Correlazione cluster-decodabilità: r={r:.3f} p={pr:.3f}')
fig, ax = plt.subplots(figsize=(6,4))
for c in (0,1): ax.scatter(np.full((labels==c).sum(), c), dec[labels==c], s=60)
ax.set_xticks([0,1]); ax.set_xticklabels(['C0','C1']); ax.set_ylabel('decodabilità (LDA)')
ax.axhline(C.CHANCE_LEVEL, color='r', ls='--', label='chance'); ax.set_title('Fenotipo vs decodabilità'); ax.legend()
plt.tight_layout(); plt.savefig(C.FIG_DIR/'pheno_vs_decodability.png', dpi=130); plt.show()

## Conclusioni
Interpreta con i numeri sopra:
- **Esistono i fenotipi?** silhouette k=2 + ARI PLV/PCC (§2-3). ARI alto = robusto.
- **Firma spaziale** (§4-5): quali regioni/coppie separano i fenotipi + permutation test (genuino?).
- **Artefatto?** (§6): se C0/C1 differiscono in gamma/varianza, la separazione è sospetta (SNR).
- **Predice la decodabilità?** (§7): è il punto che collega fenotipo → performance.

⚠️ Con 15 soggetti tutto è **esplorativo**. Se il segnale è promettente, il passo dopo è replicare
con più soggetti / la pipeline pruned esatta (vedi `prune_k` in `subject_fingerprints`).